# Week 9 - Class 12: CNN Segmentation with U-Net

**AI in Medicine and Healthcare**  
**Prof. Dr. Marcel P. Jackowski**  
**Insper - 2026**

Last week we studied well-known classical methods (thresholding, region growing, edge detection) for medical image segmentation. Today we will dive further into deep learning methods by
modifying the standard CNN, so that it can classifies pixels individually. This modification, called a U-Net, introduced back in 2015, allows a neural network to segment regions of interest.

---

## Student Information

**Student 1:** _______________  

**Student 2:** _______________  

---

## Lab Objectives

Today you'll implement CNN-based segmentation:

1. **Generate synthetic dataset** (shapes for segmentation)
2. **Build naive CNN** (see the problems!)
3. **Build U-Net** (see the solutions!)
4. **Compare architectures** (understand why U-Net works)

---

# Setup

In [ ]:
# Install packages (if needed)
# PyTorch is pre-installed in Colab
!pip install -q matplotlib scikit-learn

print('✓ Packages ready!')

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('✓ Libraries imported!')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---

# PART 1: Dataset Generation & Baseline

## Synthetic Segmentation Dataset

We'll create a **challenging** synthetic dataset to demonstrate U-Net's advantages:

**Dataset features:**
- **Task:** Segment circles and squares
- **Images:** 128×128 grayscale
- **Size:** 500 training + 150 validation/test
- **Challenges:**
  - **Varying sizes:** 15-55 pixels (multi-scale)
  - **Varying intensities:** 0.5-1.0 (appearance variation)
  - **Overlapping objects:** 3-5 per image
  - **Gaussian noise:** σ=0.15 (realistic noise)

---

In [ ]:
def create_segmentation_dataset(n_samples=500, img_size=128):
    """
    Create synthetic dataset.

    Ultra-extreme challenges:
    - EXTREME Gaussian noise (sigma=0.4)
    - EXTREME Gaussian blur (sigma=3.0)
    - EXTREME size variation (8-80 pixels)
    - MANY overlapping objects (6-10)
    - Random intensity per object (0.2-1.0)
    - Background texture/noise
    - Random transformations

    """
    from scipy.ndimage import gaussian_filter
    import numpy as np

    images = []
    masks = []

    for _ in range(n_samples):
        img = np.zeros((img_size, img_size))
        mask = np.zeros((img_size, img_size))

        # Add background texture first
        background = np.random.random((img_size, img_size)) * 0.15
        img += background

        # MANY objects (6-10) for EXTREME overlapping
        n_shapes = np.random.randint(6, 11)

        for _ in range(n_shapes):
            shape_type = np.random.choice(['circle', 'square'])

            # ULTRA-EXTREME size variation (8-80 pixels)
            if shape_type == 'circle':
                radius = np.random.randint(8, 80) // 2
                try:
                    center_x = np.random.randint(radius + 2, img_size - radius - 2)
                    center_y = np.random.randint(radius + 2, img_size - radius - 2)
                except:
                    continue

                y, x = np.ogrid[:img_size, :img_size]
                circle_mask = (x - center_x)**2 + (y - center_y)**2 <= radius**2

                # ULTRA-WIDE intensity variation (0.2-1.0)
                intensity = np.random.uniform(0.2, 1.0)
                img[circle_mask] = intensity
                mask[circle_mask] = 1

            else:  # square
                size = np.random.randint(8, 80)
                try:
                    top = np.random.randint(2, img_size - size - 2)
                    left = np.random.randint(2, img_size - size - 2)
                except:
                    continue

                intensity = np.random.uniform(0.2, 1.0)
                img[top:top+size, left:left+size] = intensity
                mask[top:top+size, left:left+size] = 1

        # EXTREME BLUR - makes boundaries very fuzzy
        img = gaussian_filter(img, sigma=3.0)

        # EXTREME NOISE - heavily disrupts patterns
        noise = np.random.normal(0, 0.4, (img_size, img_size))
        img = img + noise
        img = np.clip(img, 0, 1)

        # Add salt-and-pepper noise (sparse random pixels)
        salt_pepper = np.random.random((img_size, img_size))
        img[salt_pepper < 0.02] = 1.0  # Salt
        img[salt_pepper > 0.98] = 0.0  # Pepper

        # Random rotation
        if np.random.random() > 0.5:
            k = np.random.randint(1, 4)
            img = np.rot90(img, k)
            mask = np.rot90(mask, k)

        # Random flip
        if np.random.random() > 0.5:
            img = np.fliplr(img)
            mask = np.fliplr(mask)

        # Normalize to [0, 1]
        if img.max() > img.min():
            img = (img - img.min()) / (img.max() - img.min())

        images.append(img)
        masks.append(mask)

    images = np.array(images, dtype=np.float32)
    masks = np.array(masks, dtype=np.float32)

    return images, masks

# Generate dataset
print('Generating dataset...')
print('Features: extreme noise (σ=0.4) + extreme blur (σ=3.0) + many overlaps (6-10) + salt-pepper')
X, y = create_segmentation_dataset(n_samples=500, img_size=128)

print(f'\n✓ Dataset created!')
print(f'  Images shape: {X.shape}')
print(f'  Masks shape:  {y.shape}')
print(f'  Image range:  [{X.min():.3f}, {X.max():.3f}]')
print(f'  Mask values:  {np.unique(y)}')


In [ ]:
# Split into train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print('Dataset split:')
print(f'  Training:   {X_train.shape[0]} samples')
print(f'  Validation: {X_val.shape[0]} samples')
print(f'  Test:       {X_test.shape[0]} samples')

In [ ]:
# Visualize samples
fig, axes = plt.subplots(3, 6, figsize=(15, 8))

for i in range(3):
    # Image
    axes[i, 0].imshow(X_train[i], cmap='gray')
    axes[i, 0].set_title('Input Image' if i == 0 else '')
    axes[i, 0].axis('off')

    # Mask
    axes[i, 1].imshow(y_train[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth' if i == 0 else '')
    axes[i, 1].axis('off')

    # Overlay
    axes[i, 2].imshow(X_train[i], cmap='gray')
    axes[i, 2].imshow(y_train[i], cmap='Reds', alpha=0.3)
    axes[i, 2].set_title('Overlay' if i == 0 else '')
    axes[i, 2].axis('off')

    # More examples
    axes[i, 3].imshow(X_train[i+3], cmap='gray')
    axes[i, 3].axis('off')

    axes[i, 4].imshow(y_train[i+3], cmap='gray')
    axes[i, 4].axis('off')

    axes[i, 5].imshow(X_train[i+3], cmap='gray')
    axes[i, 5].imshow(y_train[i+3], cmap='Reds', alpha=0.3)
    axes[i, 5].axis('off')

plt.suptitle('Training Dataset Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Task: Segment the shapes (circles and squares)')
print('   Input: Grayscale image with noise')
print('   Output: Binary mask (0=background, 1=shape)')

## PyTorch Dataset & DataLoader

Create PyTorch Dataset class for efficient batch loading.

In [ ]:
class SegmentationDataset(Dataset):
    """PyTorch Dataset for segmentation."""

    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]

        # Add channel dimension: (H, W) -> (1, H, W)
        image = torch.FloatTensor(image).unsqueeze(0)
        mask = torch.FloatTensor(mask).unsqueeze(0)

        return image, mask

# Create datasets
train_dataset = SegmentationDataset(X_train, y_train)
val_dataset = SegmentationDataset(X_val, y_val)
test_dataset = SegmentationDataset(X_test, y_test)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('✓ PyTorch DataLoaders created!')
print(f'  Train batches: {len(train_loader)}')
print(f'  Val batches:   {len(val_loader)}')
print(f'  Test batches:  {len(test_loader)}')

## Evaluation Metrics

We'll use **Dice coefficient** and **IoU** to evaluate segmentation quality.

**Dice Coefficient:** $$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}$$

**IoU (Intersection over Union):** $$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

In [ ]:
def dice_coefficient(pred, target, smooth=1e-6):
    """
    Compute Dice coefficient.

    Args:
        pred: Predicted mask (B, 1, H, W)
        target: Ground truth mask (B, 1, H, W)
        smooth: Smoothing factor

    Returns:
        Dice score (scalar)
    """
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return dice.item()

def iou_score(pred, target, smooth=1e-6):
    """
    Compute IoU score.
    """
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + smooth) / (union + smooth)

    return iou.item()

def dice_loss(pred, target, smooth=1e-6):
    """
    Dice loss = 1 - Dice coefficient.
    Can be used as training loss.
    """
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return 1 - dice

print('✓ Metrics defined!')

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=20, lr=0.001):
    """
    Train a segmentation model.

    Returns:
        history: Dictionary with training history
    """
    # Define loss and optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # History
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_dice': [],
        'val_dice': []
    }

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_dice = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Metrics
            train_loss += loss.item()
            train_dice += dice_coefficient(outputs, masks)

        train_loss /= len(train_loader)
        train_dice /= len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_dice = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                outputs = model(images)
                loss = criterion(outputs, masks)

                val_loss += loss.item()
                val_dice += dice_coefficient(outputs, masks)

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)

        # Print progress
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Train Loss: {train_loss:.4f} | Dice: {train_dice:.4f} | '
              f'Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f}')

    return history

In [ ]:
# Evaluate both models on test set
def evaluate_model(model, test_loader):
    model.eval()
    total_dice = 0.0
    total_iou = 0.0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            total_dice += dice_coefficient(outputs, masks)
            total_iou += iou_score(outputs, masks)

    avg_dice = total_dice / len(test_loader)
    avg_iou = total_iou / len(test_loader)

    return avg_dice, avg_iou

---

# WARM-UP: The "Obvious" Approach

Before building proper models, let's try the most **"obvious"** idea:

## The Naive Idea:

1. **Compress** image → small representation (16×16)
2. **Classify** at small scale
3. **Expand** back to original size (128×128)

**Hypothesis:** This should work, right? We compress, classify, then expand!

**Let's test it and see what happens...** 🤔

---

In [ ]:
class CompressExpandNet(nn.Module):
    """
    Naive approach: Compress → Classify → Expand

    This is what many people think of first:
    'Just make the image smaller, classify it, then make it bigger again!'
    """

    def __init__(self):
        super(CompressExpandNet, self).__init__()

        # COMPRESS: 128×128 → 16×16 (lose spatial information)
        self.compress = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),  # 128→64
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # 64→32
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 32→16
            nn.ReLU()
        )

        # CLASSIFY at 16×16 scale (very low resolution!)
        self.classify = nn.Conv2d(128, 1, kernel_size=1)

        # EXPAND: 16×16 → 128×128 (try to recover lost info)
        self.expand = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), # 16→32
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), # 32→64
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)  # 64→128
        )

    def forward(self, x):
        # Compress (LOSE detail here!)
        x = self.compress(x)

        # Classify (at very low resolution)
        x = self.classify(x)

        # Expand (can't recover what was lost!)
        x = self.expand(x)

        # Sigmoid
        x = torch.sigmoid(x)

        return x

# Create model
model_compress_expand = CompressExpandNet().to(device)

# Count parameters
total_params = sum(p.numel() for p in model_compress_expand.parameters())
print('🔥 Compress-Expand Model Created!')
print(f'  Architecture: 128×128 → 16×16 → 128×128')
print(f'  Total parameters: {total_params:,}')

# Test forward pass
test_input = torch.randn(1, 1, 128, 128).to(device)
test_output = model_compress_expand(test_input)
print(f'\nTest: Input {test_input.shape} → Output {test_output.shape} ✓')

In [ ]:
# Train Compress-Expand model
print('Training Compress-Expand model (this will take ~5 minutes)...\n')

history_compress_expand = train_model(
    model_compress_expand,
    train_loader,
    val_loader,
    num_epochs=20,
    lr=0.001
)

print('\n✓ Compress-Expand trained!')
print(f'Final Validation Dice: {history_compress_expand["val_dice"][-1]:.4f}')
print('\n⚠️  How did it do? Let\'s visualize...')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_compress_expand['train_loss'], label='Train', linewidth=2)
axes[0].plot(history_compress_expand['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice
axes[1].plot(history_compress_expand['train_dice'], label='Train', linewidth=2)
axes[1].plot(history_compress_expand['val_dice'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Dice Score', fontsize=12)
axes[1].set_title('Dice Score', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Compress-Expand Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final Validation Dice: {history_compress_expand["val_dice"][-1]:.4f}')

In [ ]:
# Predict on test set
model_compress_expand.eval()
test_preds_compress_expand = []

with torch.no_grad():
    for i in range(len(X_test)):
        img = torch.from_numpy(X_test[i:i+1]).unsqueeze(1).to(device)
        pred = model_compress_expand(img).cpu().numpy()[0, 0]
        test_preds_compress_expand.append(pred)

test_preds_compress_expand = np.array(test_preds_compress_expand)
print(f'✓ Generated {len(test_preds_compress_expand)} predictions')
print(f'  Prediction range: [{test_preds_compress_expand.min():.3f}, {test_preds_compress_expand.max():.3f}]')

In [ ]:
# Visualize Compress-Expand results
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(3):
    # Input
    axes[i, 0].imshow(X_test[i], cmap='gray')
    axes[i, 0].set_title('Input Image' if i == 0 else '', fontsize=11)
    axes[i, 0].axis('off')

    # Ground truth
    axes[i, 1].imshow(y_test[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth' if i == 0 else '', fontsize=11)
    axes[i, 1].axis('off')

    # Compress-Expand prediction
    axes[i, 2].imshow(test_preds_compress_expand[i], cmap='gray', vmin=0, vmax=1)
    axes[i, 2].set_title('Compress-Expand\n(Very Blurry!)' if i == 0 else '',
                        fontsize=11, color='red')
    axes[i, 2].axis('off')

    # Error map
    error = np.abs(test_preds_compress_expand[i] - y_test[i])
    im = axes[i, 3].imshow(error, cmap='hot', vmin=0, vmax=1)
    axes[i, 3].set_title('Error Map\n(Red = Wrong)' if i == 0 else '', fontsize=11)
    axes[i, 3].axis('off')

plt.colorbar(im, ax=axes[:, 3], fraction=0.046, pad=0.04)
plt.suptitle('Compress-Expand Results: Why This Approach Fails!',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate metrics
dice_compress_expand, iou_compress_expand = evaluate_model(model_compress_expand, test_loader)

print('=' * 70)
print('COMPRESS-EXPAND RESULTS')
print('=' * 70)
print(f'Dice Score: {dice_compress_expand:.4f}')
print(f'IoU Score:  {iou_compress_expand:.4f}')
print('=' * 70)

---

# PART 2: Naive CNN Approach

**Naive pixel-wise classification approach:**
1. Extract features with Conv layers
2. **Classify each pixel** using 1×1 convolution
3. No multi-scale processing
4. No skip connections from early layers

Let's see this in action!

---

## Step 2.1: Build Naive CNN

**TODO:** Complete the naive CNN architecture

In [ ]:
class NaiveCNN(nn.Module):
    """
    Naive CNN: Conv layers → Pixel-wise Classification

    Problem: Limited context, no multi-scale features!
    Uses only local features to classify each pixel independently.
    """

    def __init__(self):
        super(NaiveCNN, self).__init__()

        # TODO: Define convolutional layers for feature extraction
        # Hint: Use nn.Conv2d(in_channels, out_channels, kernel_size, padding)

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        # TODO: Define pixel-wise classifier (1×1 convolution)
        # Hint: 1×1 conv acts as pixel-wise classifier
        # nn.Conv2d(64, 1, kernel_size=1)

        self.classifier = nn.Conv2d(64, 1, 1)  # 1×1 conv for pixel-wise classification

    def forward(self, x):
        # TODO: Forward pass through convolutional layers
        # Hint: Apply ReLU activation after each conv

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        # TODO: Pixel-wise classification
        # Hint: No flattening or reshaping needed!

        x = self.classifier(x)  # Classify each pixel independently

        # Sigmoid for binary segmentation
        x = torch.sigmoid(x)

        return x

# Create model
model_naive = NaiveCNN().to(device)

# Count parameters
total_params = sum(p.numel() for p in model_naive.parameters())
print(f'Naive CNN created!')
print(f'  Total parameters: {total_params:,}')

# Test forward pass
test_input = torch.randn(1, 1, 128, 128).to(device)
test_output = model_naive(test_input)
print(f'\nTest: Input {test_input.shape} → Output {test_output.shape} ✓')

## Step 2.2: Train Naive CNN

**TODO:** Write training loop for naive CNN

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=20, lr=0.001):
    """
    Train a segmentation model.

    Returns:
        history: Dictionary with training history
    """
    # Define loss and optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # History
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_dice': [],
        'val_dice': []
    }

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_dice = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)

            d_loss = 1.0 - dice_coefficient(outputs, masks)
            loss += d_loss

            # Backward pass
            loss.backward()
            optimizer.step()

            # Metrics
            train_loss += loss.item()
            train_dice += dice_coefficient(outputs, masks)

        train_loss /= len(train_loader)
        train_dice /= len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_dice = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                outputs = model(images)
                loss = criterion(outputs, masks)

                val_loss += loss.item()
                val_dice += dice_coefficient(outputs, masks)

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)

        # Print progress
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Train Loss: {train_loss:.4f} | Dice: {train_dice:.4f} | '
              f'Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f}')

    return history

# Train Naive CNN
print('Training Naive CNN (this will take 3-5 minutes)...\n')
history_naive = train_model(model_naive, train_loader, val_loader, num_epochs=20)
print('\n✓ Naive CNN trained!')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history_naive['train_loss'], label='Train', linewidth=2)
axes[0].plot(history_naive['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice
axes[1].plot(history_naive['train_dice'], label='Train', linewidth=2)
axes[1].plot(history_naive['val_dice'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Dice Score', fontsize=12)
axes[1].set_title('Dice Coefficient', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Naive CNN Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final Validation Dice: {history_naive["val_dice"][-1]:.4f}')

In [ ]:
# Predict on test set
model_naive.eval()
test_images = []
test_masks = []
test_preds_naive = []

with torch.no_grad():
    for i in range(5):
        img, mask = test_dataset[i]
        img_batch = img.unsqueeze(0).to(device)
        pred = model_naive(img_batch)

        test_images.append(img.squeeze().cpu().numpy())
        test_masks.append(mask.squeeze().cpu().numpy())
        test_preds_naive.append(pred.squeeze().cpu().numpy())

# Visualize
fig, axes = plt.subplots(5, 3, figsize=(10, 15))

for i in range(5):
    # Input
    axes[i, 0].imshow(test_images[i], cmap='gray')
    axes[i, 0].set_title('Input' if i == 0 else '')
    axes[i, 0].axis('off')

    # Ground truth
    axes[i, 1].imshow(test_masks[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth' if i == 0 else '')
    axes[i, 1].axis('off')

    # Naive prediction
    axes[i, 2].imshow(test_preds_naive[i], cmap='gray')
    axes[i, 2].set_title('Naive CNN\n(Noisy!)' if i == 0 else '')
    axes[i, 2].axis('off')

plt.suptitle('Naive CNN Predictions', fontsize=14, fontweight='bold', color='red')
plt.tight_layout()
plt.show()

---

# PART 3: U-Net Architecture

## Solution: Encoder-Decoder with Skip Connections

**U-Net architecture:**
1. **Encoder** (downsampling): Extract features at multiple scales
2. **Bottleneck**: Compressed representation
3. **Decoder** (upsampling): Reconstruct segmentation
4. **Skip connections**: Preserve spatial information!

**Why U-Net works:**
- Encoder: Learns "what" (feature extraction)
- Decoder: Learns "where" (localization)
- Skip connections: Preserve fine details
- No flattening: Maintains spatial structure throughout

---

## Step 3.1: Build U-Net

**TODO:** Complete the U-Net architecture with skip connections

In [ ]:
class UNet(nn.Module):
    """
    U-Net: Encoder-Decoder with Skip Connections

    Key innovation: Skip connections preserve spatial information!
    """

    def __init__(self):
        super(UNet, self).__init__()

        # ===== ENCODER (Downsampling) =====

        # Block 1
        self.enc1_1 = nn.Conv2d(1, 32, 3, padding=1)
        self.enc1_2 = nn.Conv2d(32, 32, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2)

        # TODO: Block 2 (64 filters)
        # Hint: Similar to block 1 but with 64 filters
        self.enc2_1 = nn.Conv2d(32, 64, 3, padding=1)
        self.enc2_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2)

        # TODO: Block 3 (128 filters)
        self.enc3_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.enc3_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2)

        # ===== BOTTLENECK =====
        self.bottleneck_1 = nn.Conv2d(128, 256, 3, padding=1)
        self.bottleneck_2 = nn.Conv2d(256, 256, 3, padding=1)

        # ===== DECODER (Upsampling) =====

        # TODO: Upsampling block 1
        # Hint: Upsample, then concatenate with enc3, then conv
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1_1 = nn.Conv2d(256 + 128, 128, 3, padding=1)  # +128 from skip connection
        self.dec1_2 = nn.Conv2d(128, 128, 3, padding=1)

        # TODO: Upsampling block 2
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2_1 = nn.Conv2d(128 + 64, 64, 3, padding=1)  # +64 from skip connection
        self.dec2_2 = nn.Conv2d(64, 64, 3, padding=1)

        # TODO: Upsampling block 3
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec3_1 = nn.Conv2d(64 + 32, 32, 3, padding=1)  # +32 from skip connection
        self.dec3_2 = nn.Conv2d(32, 32, 3, padding=1)

        # ===== OUTPUT =====
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        # ===== ENCODER =====

        # Block 1
        enc1 = F.relu(self.enc1_1(x))
        enc1 = F.relu(self.enc1_2(enc1))

        # TODO: Block 2
        enc2 = self.pool1(enc1)
        enc2 = F.relu(self.enc2_1(enc2))
        enc2 = F.relu(self.enc2_2(enc2))

        # TODO: Block 3
        enc3 = self.pool2(enc2)
        enc3 = F.relu(self.enc3_1(enc3))
        enc3 = F.relu(self.enc3_2(enc3))

        # ===== BOTTLENECK =====
        x = self.pool3(enc3)
        x = F.relu(self.bottleneck_1(x))
        x = F.relu(self.bottleneck_2(x))

        # ===== DECODER =====

        # TODO: Upsampling block 1 with skip connection
        # Hint: upsample, concatenate with enc3, conv
        x = self.up1(x)
        x = torch.cat([x, enc3], dim=1)  # Skip connection!
        x = F.relu(self.dec1_1(x))
        x = F.relu(self.dec1_2(x))

        # TODO: Upsampling block 2 with skip connection
        x = self.up2(x)
        x = torch.cat([x, enc2], dim=1)  # Skip connection!
        x = F.relu(self.dec2_1(x))
        x = F.relu(self.dec2_2(x))

        # TODO: Upsampling block 3 with skip connection
        x = self.up3(x)
        x = torch.cat([x, enc1], dim=1)  # Skip connection!
        x = F.relu(self.dec3_1(x))
        x = F.relu(self.dec3_2(x))

        # ===== OUTPUT =====
        x = self.out(x)
        x = torch.sigmoid(x)

        return x

# Create U-Net
model_unet = UNet().to(device)

# Count parameters
total_params = sum(p.numel() for p in model_unet.parameters())
print(f'U-Net created!')
print(f'  Total parameters: {total_params:,}')
print(f'\n✓ Key features:')
print('  - Encoder: Downsampling (3 levels)')
print('  - Bottleneck: Compressed representation')
print('  - Decoder: Upsampling (3 levels)')
print('  - Skip connections: Preserve spatial details')

# Test forward pass
test_input = torch.randn(1, 1, 128, 128).to(device)
test_output = model_unet(test_input)
print(f'\nTest: Input {test_input.shape} → Output {test_output.shape} ✓')

## Step 3.2: Train U-Net

**TODO:** Train U-Net using the same training function

In [ ]:
# Train U-Net
print('Training U-Net (this will take 5-10 minutes)...\n')
history_unet = train_model(model_unet, train_loader, val_loader, num_epochs=20)
print('\n✓ U-Net trained!')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history_unet['train_loss'], label='Train', linewidth=2)
axes[0].plot(history_unet['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice
axes[1].plot(history_unet['train_dice'], label='Train', linewidth=2)
axes[1].plot(history_unet['val_dice'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Dice Score', fontsize=12)
axes[1].set_title('Dice Coefficient', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('U-Net Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Final Validation Dice: {history_unet["val_dice"][-1]:.4f}')

In [ ]:
# Predict on test set
model_unet.eval()
test_preds_unet = []

with torch.no_grad():
    for i in range(5):
        img, mask = test_dataset[i]
        img_batch = img.unsqueeze(0).to(device)
        pred = model_unet(img_batch)
        test_preds_unet.append(pred.squeeze().cpu().numpy())

# Visualize
fig, axes = plt.subplots(5, 3, figsize=(10, 15))

for i in range(5):
    # Input
    axes[i, 0].imshow(test_images[i], cmap='gray')
    axes[i, 0].set_title('Input' if i == 0 else '')
    axes[i, 0].axis('off')

    # Ground truth
    axes[i, 1].imshow(test_masks[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth' if i == 0 else '')
    axes[i, 1].axis('off')

    # U-Net prediction
    axes[i, 2].imshow(test_preds_unet[i], cmap='gray')
    axes[i, 2].set_title('U-Net\n(Clean!)' if i == 0 else '')
    axes[i, 2].axis('off')

plt.suptitle('U-Net Predictions', fontsize=14, fontweight='bold', color='green')
plt.tight_layout()
plt.show()

---

# PART 4: Comparison & Analysis (15 min)

Let's compare Naive CNN vs U-Net side by side!

---

In [ ]:
# Comprehensive comparison - ALL THREE MODELS
fig, axes = plt.subplots(5, 5, figsize=(20, 16))

for i in range(5):
    # Input
    axes[i, 0].imshow(test_images[i], cmap='gray')
    axes[i, 0].set_title('Input Image' if i == 0 else '', fontsize=11)
    axes[i, 0].axis('off')

    # Ground truth
    axes[i, 1].imshow(test_masks[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth' if i == 0 else '', fontsize=11)
    axes[i, 1].axis('off')

    # Compress-Expand
    axes[i, 2].imshow(test_preds_compress_expand[i], cmap='gray', vmin=0, vmax=1)
    axes[i, 2].set_title('Compress-Expand\n(Blurry!)' if i == 0 else '',
                        fontsize=11, color='darkred')
    axes[i, 2].axis('off')

    # Naive CNN
    axes[i, 3].imshow(test_preds_naive[i], cmap='gray', vmin=0, vmax=1)
    axes[i, 3].set_title('Naive CNN\n(Better)' if i == 0 else '',
                        fontsize=11, color='orange')
    axes[i, 3].axis('off')

    # U-Net
    axes[i, 4].imshow(test_preds_unet[i], cmap='gray', vmin=0, vmax=1)
    axes[i, 4].set_title('U-Net\n(Best!)' if i == 0 else '',
                        fontsize=11, color='green')
    axes[i, 4].axis('off')

plt.suptitle('Complete Comparison: Compress-Expand → Naive CNN → U-Net',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate all three models
dice_naive, iou_naive = evaluate_model(model_naive, test_loader)
dice_unet, iou_unet = evaluate_model(model_unet, test_loader)
# dice_compress_expand already calculated

print('\n' + '='*70)
print('QUANTITATIVE COMPARISON - ALL THREE MODELS')
print('='*70)
print(f"{'Model':<20} {'Dice':<12} {'IoU':<12}")
print('-' * 70)
print(f"{'Compress-Expand':<20} {dice_compress_expand:<12.4f} {iou_compress_expand:<12.4f}")
print(f"{'Naive CNN':<20} {dice_naive:<12.4f} {iou_naive:<12.4f}")
print(f"{'U-Net':<20} {dice_unet:<12.4f} {iou_unet:<12.4f}")
print('='*70)

print('\n' + '='*70)
print('IMPROVEMENTS')
print('='*70)
print(f'Naive CNN vs Compress-Expand:')
print(f'  Dice: +{dice_naive - dice_compress_expand:.4f} ({((dice_naive - dice_compress_expand)/dice_compress_expand*100):.1f}% improvement)')
print(f'  IoU:  +{iou_naive - iou_compress_expand:.4f}')
print(f'\nU-Net vs Naive CNN:')
print(f'  Dice: +{dice_unet - dice_naive:.4f} ({((dice_unet - dice_naive)/dice_naive*100):.1f}% improvement)')
print(f'  IoU:  +{iou_unet - iou_naive:.4f}')
print(f'\nU-Net vs Compress-Expand:')
print(f'  Dice: +{dice_unet - dice_compress_expand:.4f} ({((dice_unet - dice_compress_expand)/dice_compress_expand*100):.1f}% improvement)')
print(f'  IoU:  +{iou_unet - iou_compress_expand:.4f}')
print('='*70)


## Final Reflection Questions

**1. Why does the Naive CNN produce noisy segmentations?**

**Your answer:** _______________

**2. What is the purpose of skip connections in U-Net?**

**Your answer:** _______________

**3. Compare U-Net with classical methods from last week. What are the advantages?**

**Your answer:** _______________

**4. When would you use U-Net in medical imaging?**

**Your answer:** _______________

**5. What is the role of the encoder vs decoder in U-Net?**

**Your answer:** _______________

---